# Practice 4
- More LeetCode questions

# Questions #601-625

In [ ]:
from typing import List, Optional

## 601. Triangle

- Given a triangle array, return the minimum path sum from top to bottom.

- For each step, you may move to an adjacent number of the row below. More formally, if you are on index i on the current row, you may move to either index i or index i + 1 on the next row.

 

- Example 1:
    - Input: triangle = [[2],[3,4],[6,5,7],[4,1,8,3]]
    - Output: 11
    - Explanation: The triangle looks like:
   2
  3 4
 6 5 7
4 1 8 3
The minimum path sum from top to bottom is 2 + 3 + 5 + 1 = 11 (underlined above).

- Example 2:
    - Input: triangle = [[-10]]
    - Output: -10
 

- Constraints:
    - 1 <= triangle.length <= 200
    - triangle[0].length == 1
    - triangle[i].length == triangle[i - 1].length + 1
    - -10^4 <= triangle[i][j] <= 10^4





In [ ]:
# dynamic programming: bottom up, 
# algorithm goes from bottom up and add each of the min of the prev row to the current row.
# time complexity: O(n^2)
# space complexity: O(n)
# greedy will not work. backtracking is too long
class Solution:
    def minimumTotal(self, triangle: List[List[int]]) -> int:
        dp = triangle[-1][:]
        for i in range(len(triangle) - 2, -1, -1):
            for j in range(len(triangle[i])):
                dp[j] = triangle[i][j] + min(dp[j], dp[j + 1])

        return dp[0]
    
# could do it in place and have O(1) space complexity
class Solution:
    def minimumTotal(self, triangle: List[List[int]]) -> int:
        for row in range(len(triangle) - 2, -1, -1):
            for col in range(len(triangle[row])):
                triangle[row][col] += min(triangle[row + 1][col], triangle[row + 1][col + 1])

        return triangle[0][0]
    
# top down approach with memoization
class Solution:
    def minimumTotal(self, triangle: List[List[int]]) -> int:
        memo = {}
        n = len(triangle)
        
        def dfs(row, col):
            if (row, col) in memo:
                return memo[(row, col)]
            
            if row == n - 1:
                return triangle[row][col]
            
            result = triangle[row][col] + min(dfs(row + 1, col), dfs(row + 1, col + 1))
            
            memo[(row, col)] = result 
            return result 
        
        return dfs(0, 0)
    


## 602. Wildcard Matching

- Given an input string (s) and a pattern (p), implement wildcard pattern matching with support for '?' and '*' where:
    - '?' Matches any single character.
    - '*' Matches any sequence of characters (including the empty sequence).

- The matching should cover the entire input string (not partial).

 

- Example 1:
    - Input: s = "aa", p = "a"
    - Output: false
    - Explanation: "a" does not match the entire string "aa".

- Example 2:
    - Input: s = "aa", p = "*"
    - Output: true
    - Explanation: '*' matches any sequence.

- Example 3:
    - Input: s = "cb", p = "?a"
    - Output: false
    - Explanation: '?' matches 'c', but the second letter is 'a', which does not match 'b'.
 

- Constraints:
    - 0 <= s.length, p.length <= 2000
    - s contains only lowercase English letters.
    - p contains only lowercase English letters, '?' or '*'.

In [ ]:
# dynamic programming version
# time complexity:
# time complexity: O(m * n)
# space complexity: O(m * n)
class Solution:
    def isMatch(self, s: str, p: str) -> bool:
        n = len(s)
        m = len(p)

        dp = [[False] * (m + 1) for _ in range(n + 1)]
        dp[0][0] = True

        for j in range(1, m + 1):
            if p[j - 1] == "*":
                dp[0][j] = dp[0][j - 1]

        for i in range(1, n + 1):
            for j in range(1, m + 1):
                if p[j - 1] == s[i - 1] or p[j - 1] == "?":
                    dp[i][j] = dp[i - 1][j - 1]
                elif p[j - 1] == "*":
                    dp[i][j] = dp[i][j - 1] or dp[i - 1][j]

        return dp[n][m]
    
# there is another efficient solution that use greedy with backtracking
# time complexity: avg O(m + n), worst: O(m * n)
# space complexity: O(1)
class Solution:
    def isMatch(self, s: str, p: str) -> bool:
        s_ptr, p_ptr = 0, 0
        last_s, last_p = -1, -1

        while s_ptr < len(s):
            if p_ptr < len(p) and (p[p_ptr] == '?' or p[p_ptr] == s[s_ptr]):
                s_ptr += 1
                p_ptr += 1
            elif p_ptr < len(p) and p[p_ptr] == '*':
                last_p = p_ptr
                last_s = s_ptr
                p_ptr += 1
            elif last_p != -1:
                p_ptr = last_p + 1
                last_s += 1
                s_ptr = last_s
            else:
                return False

        return all(x == '*' for x in p[p_ptr:])

## 603. Scramble String

- We can scramble a string s to get a string t using the following algorithm:

- If the length of the string is 1, stop.
- If the length of the string is > 1, do the following:
    - Split the string into two non-empty substrings at a random index, i.e., if the string is s, divide it to x and y where s = x + y.
    - Randomly decide to swap the two substrings or to keep them in the same order. i.e., after this step, s may become s = x + y or s = y + x.
    - Apply step 1 recursively on each of the two substrings x and y.
- Given two strings s1 and s2 of the same length, return true if s2 is a scrambled string of s1, otherwise, return false.

 

- Example 1:
    - Input: s1 = "great", s2 = "rgeat"
    - Output: true
    - Explanation: One possible scenario applied on s1 is:
"great" --> "gr/eat" // divide at random index.
"gr/eat" --> "gr/eat" // random decision is not to swap the two substrings and keep them in order.
"gr/eat" --> "g/r / e/at" // apply the same algorithm recursively on both substrings. divide at random index each of them.
"g/r / e/at" --> "r/g / e/at" // random decision was to swap the first substring and to keep the second substring in the same order.
"r/g / e/at" --> "r/g / e/ a/t" // again apply the algorithm recursively, divide "at" to "a/t".
"r/g / e/ a/t" --> "r/g / e/ a/t" // random decision is to keep both substrings in the same order.
The algorithm stops now, and the result string is "rgeat" which is s2.
As one possible scenario led s1 to be scrambled to s2, we return true.

- Example 2:
    - Input: s1 = "abcde", s2 = "caebd"
    - Output: false

- Example 3:
    - Input: s1 = "a", s2 = "a"
    - Output: true
 

- Constraints:
    - s1.length == s2.length
    - 1 <= s1.length <= 30
    - s1 and s2 consist of lowercase English letters.



In [ ]:
# dynamic programing
# time complexity: O(n^4)
# time complexity: O(n^3)
class Solution:
    def isScramble(self, s1: str, s2: str) -> bool:
        n = len(s1)
        dp = [[[False] * n for _ in range(n)] for _ in range(n + 1)]

        for i in range(n):
            for j in range(n):
                dp[1][i][j] = s1[i] == s2[j]

        for length in range(2, n + 1):
            for i in range(n + 1 - length):
                for j in range(n + 1 - length):
                    for new_length in range(1, length):
                        dp1 = dp[new_length][i]
                        dp2 = dp[length - new_length][i + new_length]
                        dp[length][i][j] |= dp1[j] and dp2[j + new_length]
                        dp[length][i][j] |= (
                            dp1[j + length - new_length] and dp2[j]
                        )
        return dp[n][0][0]

## 604. Unique Binary Search Trees II

- Given an integer n, return all the structurally unique BST's (binary search trees), which has exactly n nodes of unique values from 1 to n. Return the answer in any order.

 

- Example 1:
    - Input: n = 3
    - Output: [[1,null,2,null,3],[1,null,3,2],[2,1,3],[3,1,null,null,2],[3,2,null,1]]

- Example 2:
    - Input: n = 1
    - Output: [[1]]
 

- Constraints:
    - 1 <= n <= 8

In [ ]:
# Definition for a binary tree node.
class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right
# dynamic programming:
# Catalan number.
# time complexity: O(n * G(n)); G is the catalan number
# space complexity: O(n * G(n))
# very slow algorithm
class Solution:
    '''
    1) Iterate through Roots: For a range [1,n], try every number as the root of the tree.
    2) Divide: 
        The left subtree must be built using values from [start, i - 1]
        The right subtree must be built using values from [i + 1, end].
    3) Combine: Recursively generate all possible left subtrees and all possible right subtrees. 
        Then, for every combination of one left subtree and one right subtree, create a new root node with value i.
    4) Base Case: If start > end, the range is empty, so return a list containing None (representing an empty tree). 
    '''
    
    def generateTrees(self, n: int) -> List[Optional[TreeNode]]:
        # catalan number: yes it is.
        if n == 0:
            return []
        
        memo = {}
        def recurse(first, last):
            if first > last:
                return [None]
            
            if (first, last) in  memo:
                return memo[(first, last)]
            
            all_trees = []
            for i in range(first, last + 1):
                left = recurse(first, i - 1)
                right = recurse(i + 1, last)
                for l in left:
                    for r in right:
                        all_trees.append(TreeNode(i, l, r))
            memo[(first, last)] = all_trees
            return all_trees
        
        return recurse(1, n)

## 605. Palindrome Partitioning II

- Given a string s, partition s such that every substring of the partition is a palindrome.

- Return the minimum cuts needed for a palindrome partitioning of s.

 

- Example 1:
    - Input: s = "aab"
    - Output: 1
    - Explanation: The palindrome partitioning ["aa","b"] could be produced using 1 cut.

- Example 2:
    - Input: s = "a"
    - Output: 0

- Example 3:
    - Input: s = "ab"
    - Output: 1
 

- Constraints:
    - 1 <= s.length <= 2000
    - s consists of lowercase English letters only.

In [ ]:
# dynamic programming
# time complexity: O(n^2)
# space complexity: O(n^2)
class Solution:
    def minCut(self, s: str) -> int:
        n = len(s)

        palindrome = [[True] * n for _ in range(n)]

        for i in range(n - 1, -1, -1):
            for j in range(i + 1, n):
                palindrome[i][j] = s[i] == s[j] and palindrome[i + 1][j - 1]

        results = list(range(n))
        for i in range(1, n):
            for j in range(i + 1):
                if palindrome[j][i]:
                    results[i] = min(results[i], 1 + results[j - 1] if j else 0)

        return results[-1]

## 606. N-th Tribonacci Number

- The Tribonacci sequence Tn is defined as follows: 

- T0 = 0, T1 = 1, T2 = 1, and Tn+3 = Tn + Tn+1 + Tn+2 for n >= 0.

- Given n, return the value of Tn.

 

- Example 1:
    - Input: n = 4
    - Output: 4
    - Explanation:
T_3 = 0 + 1 + 1 = 2
T_4 = 1 + 1 + 2 = 4

- Example 2:
    - Input: n = 25
    - Output: 1389537
 

- Constraints:
    - 0 <= n <= 37
    - The answer is guaranteed to fit within a 32-bit integer, ie. answer <= 2^31 - 1.

- Hint 1
    - Make an array F of length 38, and set F[0] = 0, F[1] = F[2] = 1.
- Hint 2
    - Now write a loop where you set F[n+3] = F[n] + F[n+1] + F[n+2], and return F[n].

In [ ]:
# follow the hints to build a memo array for dynamic programming questions
# just be careful for out of index loop
# time complexity: O(n)
# space complexity: O(n)
class Solution:
    def tribonacci(self, n: int) -> int:
        dp = [0] * 38
        dp[0] = 0
        dp[1] = dp[2] = 1
        for i in range(n - 2):
            dp[i + 3] = dp[i] + dp[i + 1] + dp[i + 2]

        return dp[n]

## 607. Maximum Repeating Substring

- For a string sequence, a string word is k-repeating if word concatenated k times is a substring of sequence. The word's maximum k-repeating value is the highest value k where word is k-repeating in sequence. If word is not a substring of sequence, word's maximum k-repeating value is 0.

- Given strings sequence and word, return the maximum k-repeating value of word in sequence.

 

- Example 1:
    - Input: sequence = "ababc", word = "ab"
    - Output: 2
    - Explanation: "abab" is a substring in "ababc".

- Example 2:
    - Input: sequence = "ababc", word = "ba"
    - Output: 1
    - Explanation: "ba" is a substring in "ababc". "baba" is not a substring in "ababc".

- Example 3:
    - Input: sequence = "ababc", word = "ac"
    - Output: 0
    - Explanation: "ac" is not a substring in "ababc". 
 

- Constraints:
    - 1 <= sequence.length <= 100
    - 1 <= word.length <= 100
    - sequence and word contains only lowercase English letters.

- Hint 1
    - The constraints are low enough for a brute force approach.
- Hint 2
    - Try every k value from 0 upwards until word is no longer k-repeating.

In [ ]:
# brute force
# time complexity: O(n * m)
# space complexity: O(1)
class Solution:
    def maxRepeating(self, sequence: str, word: str) -> int:
        n = len(sequence) // len(word)
        for k in range(n, - 1, - 1):
            if word * k in sequence:
                return k
            
# dp solution that fits a dp table template with memo
# time complexity: O(n * m)
# space complexit: O(n)
class Soluction:
    def maxRepeating(self, sequence: str, word: str) -> int:
        n = len(sequence) 
        m = len(word)
        
        dp = [0] * n 
        
        for i in range(m - 1, n):
            if sequence[i - m + 1: i + 1] == word:
                if i >= m:
                    dp[i] = dp[i - m] + 1
                else:
                    dp[i] = 1
        return max(dp)

## 608. Maximal Square

- Given an m x n binary matrix filled with 0's and 1's, find the largest square containing only 1's and return its area.

 

- Example 1:
    - Input: matrix = [["1","0","1","0","0"],["1","0","1","1","1"],["1","1","1","1","1"],["1","0","0","1","0"]]
    - Output: 4

- Example 2:
    - Input: matrix = [["0","1"],["1","0"]]
    - Output: 1

- Example 3:
    - Input: matrix = [["0"]]
    - Output: 0
 

- Constraints:
    - m == matrix.length
    - n == matrix[i].length
    - 1 <= m, n <= 300
    - matrix[i][j] is '0' or '1'.



In [ ]:
# dynamic programming: have a table and check the max valid size and 
# multiply it by itselt to get the square.
# time complexity: O(mn)
# space complexity: O(mn)
class Solution:
    def maximalSquare(self, matrix: List[List[str]]) -> int:
        m = len(matrix)
        n = len(matrix[0])

        largest_side = 0

        dp = [[0] * (n + 1) for _ in range(m + 1)]
        for i in range(m):
            for j in range(n):
                if matrix[i][j] == "1":
                    dp[i + 1][j + 1] = min(dp[i][j + 1],
                                        dp[i + 1][j], dp[i][j]) + 1
                    largest_side = max(largest_side, dp[i + 1][j + 1])

        return largest_side * largest_side

## 609. Different Ways to Add Parentheses

- Given a string expression of numbers and operators, return all possible results from computing all the different possible ways to group numbers and operators. You may return the answer in any order.

- The test cases are generated such that the output values fit in a 32-bit integer and the number of different results does not exceed 10^4.

 

- Example 1:
    - Input: expression = "2-1-1"
    - Output: [0,2]
    - Explanation:
((2-1)-1) = 0 
(2-(1-1)) = 2

- Example 2:
    - Input: expression = "2*3-4*5"
    - Output: [-34,-14,-10,-10,10]
    - Explanation:
(2*(3-(4*5))) = -34 
((2*3)-(4*5)) = -14 
((2*(3-4))*5) = -10 
(2*((3-4)*5)) = -10 
(((2*3)-4)*5) = 10
 

- Constraints:
    - 1 <= expression.length <= 20
    - expression consists of digits and the operator '+', '-', and '*'.
    - All the integer values in the input expression are in the range [0, 99].
    - The integer values in the input expression do not have a leading '-' or '+' denoting the sign.

In [ ]:
# dynamic programming
# time complexity: O(n*2^n)
# space complexity: O(n^2 * 2^n)
class Solution:
    def diffWaysToCompute(self, expression: str) -> List[int]:
        memo = {}

        def recurse(start, end):
            if (start, end) in memo:
                return memo[(start, end)]

            results = []

            if start == end:
                results.append(int(expression[start]))
                return results

            if end - start == 1 and expression[start].isdigit():
                results.append(int(expression[start: end + 1]))
                return results

            for i in range(start, end + 1):
                if expression[i].isdigit():
                    continue

                left_results = recurse(start, i - 1)
                right_results = recurse(i + 1, end)

                for left_value in left_results:
                    for right_value in right_results:
                        if expression[i] == "+":
                            results.append(left_value + right_value)
                        elif expression[i] == "-":
                            results.append(left_value - right_value)
                        elif expression[i] == "*":
                            results.append(left_value * right_value)
            memo[(start, end)] = results

            return results

        return recurse(0, len(expression) - 1)

## 610. Perfect Squares

- Given an integer n, return the least number of perfect square numbers that sum to n.

- A perfect square is an integer that is the square of an integer; in other words, it is the product of some integer with itself. For example, 1, 4, 9, and 16 are perfect squares while 3 and 11 are not.

 

- Example 1:
    - Input: n = 12
    - Output: 3
    - Explanation: 12 = 4 + 4 + 4.

- Example 2:
    - Input: n = 13
    - Output: 2
    - Explanation: 13 = 4 + 9.
 

- Constraints:
    - 1 <= n <= 10^4

In [ ]:
# dynamic programming: coin change with perfect square twist
# time complexity: O(n*k), k is the amount
# space complexity: O(n)
# there are a couple optimization to make it a lot faster, but I think
# the author intend to have this as a dp - coin change problem.
class Solution:
    def numSquares(self, n: int) -> int:
        # classic coin change with the perfect square twist
        perfect_square = set()
        for i in range(1, n + 1):
            square = i * i
            if square <= n:
                perfect_square.add(square)
        
        arr = list(perfect_square)
        # print(arr)
        dp = [float("inf")] * (n + 1)
        dp[0] = 0
        for i in range(1, n + 1):
            for num in arr:
                if i - num >= 0:
                    dp[i] = min(dp[i], dp[i  - num] + 1)
        
        return dp[n] if dp[n] != float('inf') else -1

## 611. Word Break II

- Given a string s and a dictionary of strings wordDict, add spaces in s to construct a sentence where each word is a valid dictionary word. Return all such possible sentences in any order.

- Note that the same word in the dictionary may be reused multiple times in the segmentation.

 

- Example 1:
    - Input: s = "catsanddog", wordDict = ["cat","cats","and","sand","dog"]
    - Output: ["cats and dog","cat sand dog"]

- Example 2:
    - Input: s = "pineapplepenapple", wordDict = ["apple","pen","applepen","pine","pineapple"]
    - Output: ["pine apple pen apple","pineapple pen apple","pine applepen apple"]
    - Explanation: Note that you are allowed to reuse a dictionary word.

- Example 3:
    - Input: s = "catsandog", wordDict = ["cats","dog","sand","and","cat"]
    - Output: []
 

- Constraints:
    - 1 <= s.length <= 20
    - 1 <= wordDict.length <= 1000
    - 1 <= wordDict[i].length <= 10
    - s and wordDict[i] consist of only lowercase English letters.
    - All the strings of wordDict are unique.
    - Input is generated in a way that the length of the answer doesn't exceed 105.

In [ ]:
# dynamic programming: using recursion and memoization
# time complexity: O(2^n)
# time complexity: O(2^n)
class Solution:
    def wordBreak(self, s: str, wordDict: List[str]) -> List[str]:
        word_set = set(wordDict)

        memo = {}

        def recurse(cur_s):
            if cur_s in memo:
                return memo[cur_s]

            if not cur_s:
                return [""]

            res = []

            for i in range(1, len(cur_s) + 1):
                word = cur_s[:i]
                if word in word_set:
                    suffix_ways = recurse(cur_s[i:])

                    for way in suffix_ways:
                        if way == "":
                            res.append(word)
                        else:
                            res.append(word + " " + way)
            memo[cur_s] = res
            return res

        return recurse(s)

## 612. N-Queens II

- The n-queens puzzle is the problem of placing n queens on an n x n chessboard such that no two queens attack each other.

- Given an integer n, return the number of distinct solutions to the n-queens puzzle.

 

- Example 1:
    - Input: n = 4
    - Output: 2
    - Explanation: There are two distinct solutions to the 4-queens puzzle as shown.

- Example 2:
    - Input: n = 1
    - Output: 1
 

- Constraints:
    - 1 <= n <= 9

In [ ]:
# backtracking
# time complexity: O(n!)
# space complexity: O(n)
class Solution:
    def totalNQueens(self, n: int) -> int:
        result = 0
        cols = [False] * 10
        dg = [False] * 20
        udg = [False] * 20

        def backtrack(i):
            if i == n:
                nonlocal result
                result += 1
                return
            for j in range(n):
                a, b = i + j, i - j + n
                if cols[j] or dg[a] or udg[b]:
                    continue
                cols[j] = dg[a] = udg[b] = True
                backtrack(i + 1)
                cols[j] = dg[a] = udg[b] = False

        backtrack(0)
        return result

## 613. Spiral Matrix II

- Given a positive integer n, generate an n x n matrix filled with elements from 1 to n2 in spiral order.

 

- Example 1:
    - Input: n = 3
    - Output: [[1,2,3],[8,9,4],[7,6,5]]

- Example 2:
    - Input: n = 1
    - Output: [[1]]
 

- Constraints:
    - 1 <= n <= 20

In [ ]:
# matrix manipulation
# time complexity: O(n^2)
# space complexity: O(n^2)
class Solution:
    def generateMatrix(self, n: int) -> List[List[int]]:
        result = [[0] * n for _ in range(n)]
        directions = (0, 1, 0, -1, 0)
        i = j = k = 0
        total_square = n * n + 1
        for v in range(1,total_square):
            result[i][j] = v
            x, y = i + directions[k], j + directions[k + 1]
            if x < 0 or x >= n or y < 0 or y >= n or result[x][y]:
                k = (k + 1) % 4
            i, j = i + directions[k], j + directions[k + 1]
            
        return result

## 614. Maximize the Distance Between Points on a Square

- You are given an integer side, representing the edge length of a square with corners at (0, 0), (0, side), (side, 0), and (side, side) on a Cartesian plane.

- You are also given a positive integer k and a 2D integer array points, where points[i] = [xi, yi] represents the coordinate of a point lying on the boundary of the square.

- You need to select k elements among points such that the minimum Manhattan distance between any two points is maximized.

- Return the maximum possible minimum Manhattan distance between the selected k points.

- The Manhattan Distance between two cells (xi, yi) and (xj, yj) is |xi - xj| + |yi - yj|.

 

- Example 1:
    - Input: side = 2, points = [[0,2],[2,0],[2,2],[0,0]], k = 4
    - Output: 2
    - Explanation:
Select all four points.

- Example 2:
    - Input: side = 2, points = [[0,0],[1,2],[2,0],[2,2],[2,1]], k = 4
    - Output: 1
    - Explanation:
Select the points (0, 0), (2, 0), (2, 2), and (2, 1).

- Example 3:
    - Input: side = 2, points = [[0,0],[0,1],[0,2],[1,2],[2,0],[2,2],[2,1]], k = 5
    - Output: 1
    - Explanation:
Select the points (0, 0), (0, 1), (0, 2), (1, 2), and (2, 2).

 

- Constraints:
    - 1 <= side <= 10^9
    - 4 <= points.length <= min(4 * side, 15 * 10^3)
    - points[i] == [xi, yi]
    - The input is generated such that:
    - points[i] lies on the boundary of the square.
    - All points[i] are unique.
    - 4 <= k <= min(25, points.length)

- Hint 1
    - Can we use binary search for this problem?
- Hint 2
    - Think of the coordinates on a straight line in clockwise order.
- Hint 3
    - Binary search on the minimum Manhattan distance x.
- Hint 4
    - During the binary search, for each coordinate, find the immediate next coordinate with distance >= x.
- Hint 5
    - Greedily select up to k coordinates.

In [ ]:
# binary search, greedy
# time complexity: O(nlogn)
# space complexity: O(n)
from bisect import bisect_left
class Solution:
    def maxDistance(self, side: int, points: List[List[int]], k: int) -> int:
        nums = []
        for x, y in points:
            if x == 0:
                nums.append(y)
            elif y == side:
                nums.append(side + x)
            elif x == side:
                nums.append(side * 3 - y)
            else:
                nums.append(side * 4 - x)

        def check(low):
            n = len(nums)
            for num in nums:
                end = num + side * 4 - low
                cur = num
                ok = True
                for _ in range(k - 1):
                    j = bisect_left(nums, cur + low)
                    if j == n or nums[j] > end:
                        ok = False
                        break
                    cur = nums[j]
                if ok:
                    return True
            return False

        nums.sort()

        left, right = 1, side
        while left < right:
            mid = (left + right + 1) // 2
            if check(mid):
                left = mid
            else:
                right = mid - 1
        return left

## 615.

## 616.

## 617.

## 618.

## 619.

## 620.

## 621.

## 622.

## 623.

## 624.

## 625.

# Questions #626-650

## 626.

## 627.

## 628.

## 629.

## 630.

## 631.

## 632.

## 633.

## 634.

## 635.

## 636.

## 637.

## 638.

## 639.

## 640.

## 641.

## 642.

## 643.

## 644.

## 645.

## 646.

## 647.

## 648.

## 649.

## 650.

# Question #651-675

## 651.

## 652.

## 653.

## 654.

## 655.

## 656.

## 657.

## 658.

## 659.

## 660.

## 661.

## 662.

## 663.

## 664.

## 665.

## 666.

## 667.

## 668.

## 669.

## 670.

## 671.

## 672.

## 673.

## 674.

## 675.

# Question #676-700

## 676.

## 677.

## 678.

## 679.

## 680.

## 681.

## 682.

## 683.

## 684.

## 685.

## 686.

## 687.

## 688.

## 689.

## 690.

## 691.

## 692.

## 693.

## 694.

## 695.

## 696.

## 697.

## 698.

## 699.

## 700.

# Question #701-725

## 701.

## 702.

## 703.

## 704.

## 705.

## 706.

## 707.

## 708.

## 709.

## 710.

## 711.

## 712.

## 713.

## 714.

## 715.

## 716.

## 717.

## 718.

## 719.

## 720.

## 721.

## 722.

## 723.

## 724.

## 725.

# Question #726-750

## 726.

## 727.

## 728.

## 729.

## 730.

## 731.

## 732.

## 733.

## 734.

## 735.

## 736.

## 737.

## 738.

## 739.

## 740.

## 741.

## 742.

## 743.

## 744.

## 745.

## 746.

## 747.

## 748.

## 749.

## 750.

# Question #751-775

## 751.

## 752.

## 753.

## 754.

## 755.

## 756.

## 757.

## 758.

## 759.

## 760.

## 761

## 762.

## 763.

## 764.

## 765.

## 766.

## 767.

## 768.

## 769.

## 770.

## 771.

## 772.

## 773.

## 774.

## 775.

# Questions 776-800

## 776.

## 777.

## 778.

## 779.

## 780.

## 781.

## 782.

## 783.

## 784.

## 785.

## 786.

## 787.

## 788.

## 789.

## 790.

## 791.

## 792.

## 793.

## 794.

## 795.

## 796.

## 797.

## 798.

## 799.

## 800.